<a href="https://colab.research.google.com/github/SriSurya06/ai-student-portfolio/blob/main/Day2_LabA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic

In [3]:
import os
import getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass("Gemini API Key: ")


Gemini API Key: ··········


In [4]:
from pydantic import BaseModel
from typing import List, Optional


class Education(BaseModel):
    degree: str
    institution: str
    year: int


class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [5]:
from google import genai
from pydantic import ValidationError

client = genai.Client(
    api_key=os.environ['GEMINI_API_KEY']
)


def extract_resume(
    raw_text: str,
    max_retries: int = 1
):

    if not raw_text.strip():
        raise ValueError(
            "Resume text is empty"
        )

    for attempt in range(
        max_retries + 1
    ):

        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f"""
Extract a Resume JSON from this text.
Return ONLY JSON.

{raw_text}
""",
                config={
                    'response_mime_type':
                        'application/json',

                    'response_schema':
                        Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(
                resp.text
            )

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = (
                f"Fix this JSON "
                f"to match schema. "
                f"Errors: {e}. "
                f"Original: {resp.text}"
            )

            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=fix_prompt,
                config={
                    'response_mime_type':
                        'application/json',

                    'response_schema':
                        Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(
                resp.text
            )

In [6]:
import os
print(os.listdir())

['.config', 'Resume.txt', 'Resume1.txt', 'Resume2.txt', 'sample_data']


In [7]:
resume_files = [
    'Resume.txt',
    'Resume1.txt',
    'Resume2.txt'
]

resumes = []

for file in resume_files:
    with open(
        file,
        'r',
        encoding='utf-8'
    ) as f:

        resumes.append(
            f.read()
        )

print(
    f"Loaded {len(resumes)} resumes"
)

Loaded 3 resumes


In [8]:
results = []

for i, r in enumerate(resumes):

    try:
        parsed = extract_resume(r)

        results.append(parsed)

        print(
            f"Resume {i+1}: "
            f"{parsed.name} — "
            f"{len(parsed.skills)} skills, "
            f"{parsed.experience_years} years exp"
        )

    except Exception as e:

        print(
            f"Resume {i+1}: FAILED — "
            f"{type(e).__name__}: {e}"
        )


Resume 1: Rama Pithani — 17 skills, 0.0 years exp
Resume 2: Candidate Name — 10 skills, 0.0 years exp
Resume 3: Unknown Candidate — 14 skills, 0.0 years exp


In [9]:
for i, r in enumerate(results):
  print(f"\n===== Resume {i+1} =====")
  print(r.model_dump_json(indent=2))


===== Resume 1 =====
{
  "name": "Rama Pithani",
  "email": "ramapithani154@gmail.com",
  "phone": "+91-7036136496",
  "education": [
    {
      "degree": "B.Tech in Computer Science and Engineering",
      "institution": "Aditya College of Engineering and Technology, Surampalem",
      "year": 2026
    },
    {
      "degree": "Class XII",
      "institution": "Sri Chaitanya College, Kakinada",
      "year": 2023
    }
  ],
  "skills": [
    "C++",
    "C",
    "HTML",
    "CSS",
    "Java Script",
    "Basic React",
    "Basic- Node.js",
    "Express",
    "Postman",
    "SQL",
    "MongoDB",
    "Github",
    "Docker",
    "Data Structures and Algorithms",
    "OOPS",
    "Operating Systems",
    "Database Management Systems"
  ],
  "projects": [
    "JobLens– Centralized Campus Drive Management System",
    "Library Management System API"
  ],
  "experience_years": 0.0
}

===== Resume 2 =====
{
  "name": "Candidate Name",
  "email": "candidate@example.com",
  "phone": null,
  "ed

In [10]:
try:
  bad = extract_resume('')
  print("Unexpected success")
except Exception as e:
  print("Caught gracefully:", type(e).__name__)
  print("Message:", e)

Caught gracefully: ValueError
Message: Resume text is empty


## Day 2 Lab 2B — Errors handled
1. Missing phone number Handled using Optional[str] = None.
2. Invalid JSON output Retry prompt repairs malformed JSON.
3. Empty input Validation prevents processing blank resumes.
## Sample résumés processed: 3 / 3 successful\